# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 14: Hashing Seguro de Contraseñas (Bcrypt / Argon2)
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Cifrado manual con Salting y Factor de Costo*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Hashing con passlib / password_hash() de PHP* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas contra ataques de Rainbow Tables y Timing*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion14.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [2]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  Could not wait for server fd - select (11: Resource temporarily unavailable) [IP: 185.125.189.186 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


In [5]:
!pip install -q bcrypt "passlib[bcrypt]"

---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Cifrado manual con Salting y Factor de Costo

In [9]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
from passlib.hash import bcrypt

app = FastAPI(title="API Artesano - Sesión 14")

class PayloadRequest(BaseModel):
    dato: str

class UserRequest(BaseModel):
    username: str
    password: str


@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}


@app.post("/api/registro-manual")
def registro_manual(user: UserRequest):

    hashed_password = bcrypt.hash(user.password)

    return {
        "status": "success",
        "username": user.username,
        "hash_guardado": hashed_password,
        "mensaje": "Usuario registrado con cifrado manual (Salt y Costo)"
    }

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [ ]:
%%writefile app_ia.py
from fastapi import FastAPI

# Pega aquí el código generado por la IA para auditarlo


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas contra ataques de Rainbow Tables y Timing

In [10]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 14 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


=== AUDITORIA CRITICA WEB: SESION 14 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.
